In [22]:
import torch
import torch.nn as nn

In [23]:
import torch.optim as optim

In [24]:
path = r"C:/Users/user/Desktop/Myproject/Dataset/Fruit-Images-Dataset"
print("dataset root:", path)


dataset root: C:/Users/user/Desktop/Myproject/Dataset/Fruit-Images-Dataset


In [25]:
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import numpy as np


In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(777)

if device == 'cuda':
    torch.cuda.manual_seed_all(777)
    print('cuda')

cuda


In [27]:
# 데이터셋 경로 탐색 및 사용할 클래스 선택
train_candidates = [
    "fruits-360/Training",
    "Training",
    "fruits-360_dataset/Training",
]
test_candidates = [
    "fruits-360/Test",
    "Test",
    "fruits-360_dataset/Test",
]
base = Path(path)
train_root = next((base / c for c in train_candidates if (base / c).exists()), None)
test_root = next((base / c for c in test_candidates if (base / c).exists()), None)

if train_root is None or test_root is None:
    raise FileNotFoundError(f"Training/Test folders not found under {base}")

# 원하는 클래스만 지정 (추가/변경 가능)
allowed_classes = [
    "Apple Red 1",
    "Avocado",
    "Blueberry",
    
]
print(f"train_root: {train_root}")
print(f"test_root: {test_root}")
print(f"selected classes: {allowed_classes}")


train_root: C:\Users\user\Desktop\Myproject\Dataset\Fruit-Images-Dataset\Training
test_root: C:\Users\user\Desktop\Myproject\Dataset\Fruit-Images-Dataset\Test
selected classes: ['Apple Red 1', 'Avocado', 'Blueberry']


In [28]:
# RGB 평균값 + 데이터 증강(Augmentation)을 적용하는 커스텀 Dataset
class RGBFruitDataset(Dataset):
    def __init__(self, root_dir, allowed=None, augment=True, samples_per_image=5):
        """
        augment: 데이터 증강 적용 여부
        samples_per_image: 이미지당 생성할 RGB 샘플 수 (증강으로 여러 샘플 생성)
        """
        self.root = Path(root_dir)
        self.augment = augment
        self.samples_per_image = samples_per_image
        
        class_dirs = sorted([d for d in self.root.iterdir() if d.is_dir()])
        if allowed:
            class_dirs = [d for d in class_dirs if d.name in allowed]
        self.classes = [d.name for d in class_dirs]
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.image_paths = []
        self.labels = []
        
        for d in class_dirs:
            label = self.class_to_idx[d.name]
            for img_path in d.glob("*"):
                if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.image_paths.append(img_path)
                    self.labels.append(label)
        
        if not self.image_paths:
            raise RuntimeError("No images found for the selected classes.")
        
        # 증강으로 데이터 크기 증가
        if augment:
            self.samples_per_image = samples_per_image
            print(f"원본 이미지 수: {len(self.image_paths)}")
            print(f"이미지당 샘플 수: {samples_per_image}")
            print(f"총 데이터셋 크기: {len(self.image_paths) * samples_per_image}")
        else:
            self.samples_per_image = 1

    def __len__(self):
        return len(self.image_paths) * self.samples_per_image

    def _extract_rgb_with_augmentation(self, img, sample_idx):
        """이미지에서 여러 방식으로 RGB 값 추출 (데이터 증강)"""
        arr = np.array(img, dtype=np.float32)
        h, w = arr.shape[:2]
        
        # 여러 증강 기법 중 하나를 선택
        if self.augment:
            augment_type = sample_idx % 5
            
            if augment_type == 0:
                # 원본 이미지 중앙 RGB
                crop_h, crop_w = int(h*0.7), int(w*0.7)
                y1 = (h - crop_h) // 2
                x1 = (w - crop_w) // 2
                cropped = arr[y1:y1+crop_h, x1:x1+crop_w]
                rgb = cropped.reshape(-1, 3).mean(axis=0)
            
            elif augment_type == 1:
                # 랜덤 크롭 (좌상단)
                crop_h, crop_w = int(h*0.6), int(w*0.6)
                y1 = np.random.randint(0, h - crop_h)
                x1 = np.random.randint(0, w - crop_w)
                cropped = arr[y1:y1+crop_h, x1:x1+crop_w]
                rgb = cropped.reshape(-1, 3).mean(axis=0)
            
            elif augment_type == 2:
                # 밝기 조정 (±20%)
                brightness = 0.8 + np.random.rand() * 0.4  # 0.8~1.2
                arr_bright = arr * brightness
                arr_bright = np.clip(arr_bright, 0, 255)
                rgb = arr_bright.reshape(-1, 3).mean(axis=0)
            
            elif augment_type == 3:
                # 대비 조정
                contrast = 0.8 + np.random.rand() * 0.4  # 0.8~1.2
                mean_val = arr.mean()
                arr_contrast = (arr - mean_val) * contrast + mean_val
                arr_contrast = np.clip(arr_contrast, 0, 255)
                rgb = arr_contrast.reshape(-1, 3).mean(axis=0)
            
            else:  # augment_type == 4
                # 노이즈 추가
                noise = np.random.randn(*arr.shape) * 15
                arr_noisy = arr + noise
                arr_noisy = np.clip(arr_noisy, 0, 255)
                rgb = arr_noisy.reshape(-1, 3).mean(axis=0)
        else:
            # 증강 없음 - 단순 평균
            rgb = arr.reshape(-1, 3).mean(axis=0)
        
        return rgb / 255.0  # 0~1 정규화

    def __getitem__(self, idx):
        # 어느 이미지의 몇 번째 샘플인지 계산
        img_idx = idx // self.samples_per_image
        sample_idx = idx % self.samples_per_image
        
        img_path = self.image_paths[img_idx]
        label = self.labels[img_idx]
        
        img = Image.open(img_path).convert("RGB")
        rgb_normalized = self._extract_rgb_with_augmentation(img, sample_idx)
        
        features = torch.tensor(rgb_normalized, dtype=torch.float32)
        target = torch.tensor(label, dtype=torch.long)
        return features, target


In [29]:
# DataLoader 구성 (데이터 증강 활성화)
batch_size = 256

# 증강 적용: 이미지당 5개 샘플 생성 → 데이터 5배 증가
train_ds = RGBFruitDataset(train_root, allowed_classes, augment=True, samples_per_image=5)
# 테스트셋은 증강 없음 (정확한 평가를 위해)
test_ds = RGBFruitDataset(test_root, allowed_classes, augment=False, samples_per_image=1)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

num_classes = len(train_ds.classes)
print(f"classes ({num_classes}): {train_ds.classes}")


원본 이미지 수: 1381
이미지당 샘플 수: 5
총 데이터셋 크기: 6905
classes (3): ['Apple Red 1', 'Avocado', 'Blueberry']


In [30]:
# MLP 모델 정의 (RGB 3차원 입력)
class RGBFruitMLP(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = RGBFruitMLP(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)


In [31]:
# 평가 함수

def evaluate(loader):
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            out = model(x)
            loss = criterion(out, y)
            loss_sum += loss.item() * y.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return loss_sum / total, correct / total

# 학습 루프 (데이터 증강으로 더 많은 에포크 학습)
num_epochs = 10  # 에포크 증가 (데이터 5배 증가했으므로)
train_losses = []
val_losses = []
val_accs = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * y.size(0)

    train_loss = running_loss / len(train_ds)
    val_loss, val_acc = evaluate(test_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc*100:.2f}%")


Epoch 01 | train_loss 1.0591 | val_loss 0.9951 | val_acc 52.28%
Epoch 05 | train_loss 0.4376 | val_loss 0.3388 | val_acc 100.00%
Epoch 10 | train_loss 0.1549 | val_loss 0.0693 | val_acc 100.00%


In [32]:
# 학습 완료 후 저장
state = {
    "model_state": model.state_dict(),
    "classes": train_ds.classes,
    "class_to_idx": train_ds.class_to_idx,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "val_accs": val_accs,
}
torch.save(state, "rgb_fruit_mlp.pth")
print("model saved -> rgb_fruit_mlp.pth")


model saved -> rgb_fruit_mlp.pth


In [33]:
# 추가 평가: 전체 정확도와 클래스별 정확도
model.eval()
correct = 0
total = 0
per_class_correct = [0 for _ in range(num_classes)]
per_class_total = [0 for _ in range(num_classes)]

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)
        for t, p in zip(y, preds):
            per_class_total[t.item()] += 1
            if t == p:
                per_class_correct[t.item()] += 1

overall_acc = 100.0 * correct / total
print(f"Test Accuracy: {overall_acc:.2f}%")

print("Per-class Accuracy:")
for cls, c_total, c_correct in zip(train_ds.classes, per_class_total, per_class_correct):
    acc = 100.0 * c_correct / c_total if c_total > 0 else 0.0
    print(f"  {cls:20s}: {acc:5.2f}% ({c_correct}/{c_total})")


Test Accuracy: 100.00%
Per-class Accuracy:
  Apple Red 1         : 100.00% (164/164)
  Avocado             : 100.00% (143/143)
  Blueberry           : 100.00% (154/154)


In [13]:
# 아두이노 시리얼 통신용 패키지 설치
!pip install pyserial


In [34]:
import serial
import time

SERIAL_PORT = 'COM4'  # 아두이노 포트 확인 후 수정
BAUD_RATE = 9600

# 저장된 모델 불러오기
checkpoint = torch.load("rgb_fruit_mlp.pth", map_location=device)
loaded_model = RGBFruitMLP(num_classes=len(checkpoint['classes'])).to(device)
loaded_model.load_state_dict(checkpoint['model_state'])
loaded_model.eval()

classes = checkpoint['classes']
print(f"Model loaded. Classes: {classes}")


Model loaded. Classes: ['Apple Red 1', 'Avocado', 'Blueberry']


In [35]:
# 아두이노와 실시간 통신 (무한 루프)
# 아두이노에서 "R,G,B" 형식으로 전송하면 추론 결과 반환
# 예: "120,200,50" → PC가 추론 → "Orange" 출력

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    time.sleep(2)  # 아두이노 초기화 대기
    print(f"Connected to {SERIAL_PORT} at {BAUD_RATE} baud")
    print("Waiting for RGB data from Arduino...")
    print("Press Ctrl+C to stop.\n")
    
    while True:
        if ser.in_waiting > 0:
            line = ser.readline().decode('utf-8').strip()
            print(f"Received: {line}")
            
            try:
                # RGB 값 파싱 (여러 형식 지원)
                # 형식1: "120,200,50" (CSV)
                # 형식2: "R: 120   G: 200   B: 50" (디버그 형식)
                
                if ',' in line and 'R:' not in line:
                    # CSV 형식
                    r, g, b = map(float, line.split(','))
                else:
                    # 디버그 형식 파싱
                    parts = line.split()
                    r = float(parts[parts.index('R:') + 1]) if 'R:' in parts else 0
                    g = float(parts[parts.index('G:') + 1]) if 'G:' in parts else 0
                    b = float(parts[parts.index('B:') + 1]) if 'B:' in parts else 0
                
                # 0~255 → 0~1 정규화
                rgb_normalized = torch.tensor([r/255.0, g/255.0, b/255.0], dtype=torch.float32).to(device)
                
                # 모델 추론
                with torch.no_grad():
                    logits = loaded_model(rgb_normalized.unsqueeze(0))
                    pred_idx = logits.argmax(1).item()
                    pred_class = classes[pred_idx]
                    confidence = torch.softmax(logits, dim=1)[0][pred_idx].item() * 100
                
                result = f"Prediction: {pred_class} ({confidence:.2f}%)"
                print(result)
                
                # 아두이노로 결과 전송 (선택)
                ser.write(f"{pred_class}\n".encode('utf-8'))
                print("-" * 50)
                
            except ValueError:
                print("Invalid format. Expected 'R,G,B' (e.g., 120,200,50)")
        
        time.sleep(0.1)

except serial.SerialException as e:
    print(f"Serial error: {e}")
    print(f"Please check if Arduino is connected to {SERIAL_PORT}")
except KeyboardInterrupt:
    print("\nStopped by user")
finally:
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print("Serial port closed")


Connected to COM4 at 9600 baud
Waiting for RGB data from Arduino...
Press Ctrl+C to stop.

Received: TCS34725 Ready
Prediction: Blueberry (74.85%)
--------------------------------------------------
Received: 3,4,4
Prediction: Blueberry (76.83%)
--------------------------------------------------
Received: Result: Blueberry
Prediction: Blueberry (74.85%)
--------------------------------------------------
Received: 3,4,3
Prediction: Blueberry (74.24%)
--------------------------------------------------
Received: Result: Blueberry
Prediction: Blueberry (74.85%)
--------------------------------------------------
Received: 37,13,10
Prediction: Avocado (90.13%)
--------------------------------------------------
Received: Result: Blueberry
Prediction: Blueberry (74.85%)
--------------------------------------------------
Received: 24,6,6
Prediction: Avocado (75.23%)
--------------------------------------------------
Received: Result: Blueberry
Prediction: Blueberry (74.85%)
---------------------

In [36]:
# 학습 데이터의 실제 RGB 평균값 확인
print("=== 학습 데이터의 클래스별 RGB 평균값 ===\n")
class_rgb_stats = {}

for cls_name in train_ds.classes:
    cls_idx = train_ds.class_to_idx[cls_name]
    rgb_values = []
    
    for img_path, label in train_ds.samples:
        if label == cls_idx:
            img = Image.open(img_path).convert("RGB")
            arr = np.array(img, dtype=np.float32).reshape(-1, 3)
            rgb_mean = arr.mean(axis=0)
            rgb_values.append(rgb_mean)
    
    rgb_array = np.array(rgb_values)
    mean_rgb = rgb_array.mean(axis=0)
    std_rgb = rgb_array.std(axis=0)
    
    class_rgb_stats[cls_name] = {
        'mean': mean_rgb,
        'std': std_rgb
    }
    
    print(f"{cls_name}:")
    print(f"  평균 RGB: R={mean_rgb[0]:.1f}, G={mean_rgb[1]:.1f}, B={mean_rgb[2]:.1f}")
    print(f"  표준편차: R={std_rgb[0]:.1f}, G={std_rgb[1]:.1f}, B={std_rgb[2]:.1f}")
    print()


=== 학습 데이터의 클래스별 RGB 평균값 ===



AttributeError: 'RGBFruitDataset' object has no attribute 'samples'

In [ ]:
# 수동 테스트: RGB 값 직접 입력해서 예측 확인
def test_rgb_prediction(r, g, b):
    """RGB 값을 입력하면 모델이 어떻게 예측하는지 확인"""
    rgb_input = torch.tensor([r/255.0, g/255.0, b/255.0], dtype=torch.float32).to(device)
    
    with torch.no_grad():
        logits = loaded_model(rgb_input.unsqueeze(0))
        probs = torch.softmax(logits, dim=1)[0]
        pred_idx = logits.argmax(1).item()
        
        print(f"\nInput RGB: R={r}, G={g}, B={b}")
        print("예측 확률:")
        for i, cls in enumerate(classes):
            print(f"  {cls:20s}: {probs[i].item()*100:5.2f}%")
        print(f"\n최종 예측: {classes[pred_idx]}")
        print("-" * 50)

# 예시 테스트
print("=== 수동 테스트 ===")
test_rgb_prediction(200, 50, 50)   # 빨간색 (Apple Red 1)
test_rgb_prediction(100, 150, 50)  # 녹색 (Avocado)
test_rgb_prediction(50, 50, 200)   # 파란색 (Blueberry)


=== 수동 테스트 ===

Input RGB: R=250, G=230, B=50
예측 확률:
  Apple Red 1         : 99.95%
  Avocado             :  0.05%
  Blueberry           :  0.00%

최종 예측: Apple Red 1
--------------------------------------------------

Input RGB: R=200, G=50, B=50
예측 확률:
  Apple Red 1         : 99.98%
  Avocado             :  0.02%
  Blueberry           :  0.00%

최종 예측: Apple Red 1
--------------------------------------------------

Input RGB: R=50, G=50, B=200
예측 확률:
  Apple Red 1         :  0.00%
  Avocado             :  0.01%
  Blueberry           : 99.99%

최종 예측: Blueberry
--------------------------------------------------

Input RGB: R=255, G=150, B=50
예측 확률:
  Apple Red 1         : 99.99%
  Avocado             :  0.01%
  Blueberry           :  0.00%

최종 예측: Apple Red 1
--------------------------------------------------


In [ ]:
# 학습 데이터의 원본 이미지(증강 전)에서 RGB 값 확인
print("=== 원본 이미지의 클래스별 RGB 통계 (증강 제외) ===\n")

for cls_name in train_ds.classes:
    cls_idx = train_ds.class_to_idx[cls_name]
    rgb_values = []
    
    # 원본 이미지만 처리 (증강 없이)
    for img_path, label in zip(train_ds.image_paths, train_ds.labels):
        if label == cls_idx:
            img = Image.open(img_path).convert("RGB")
            arr = np.array(img, dtype=np.float32).reshape(-1, 3)
            rgb_mean = arr.mean(axis=0)
            rgb_values.append(rgb_mean)
    
    if rgb_values:
        rgb_array = np.array(rgb_values)
        mean_rgb = rgb_array.mean(axis=0)
        std_rgb = rgb_array.std(axis=0)
        min_rgb = rgb_array.min(axis=0)
        max_rgb = rgb_array.max(axis=0)
        
        print(f"{cls_name}:")
        print(f"  평균 RGB: ({mean_rgb[0]:.0f}, {mean_rgb[1]:.0f}, {mean_rgb[2]:.0f})")
        print(f"  범위: R[{min_rgb[0]:.0f}~{max_rgb[0]:.0f}], G[{min_rgb[1]:.0f}~{max_rgb[1]:.0f}], B[{min_rgb[2]:.0f}~{max_rgb[2]:.0f}]")
        print()

# 모델이 현재 예측하는 결과와 비교
print("\n=== 모델 예측 vs 학습 데이터 비교 ===")
test_cases = [
    ("빨간색(Apple Red 1)", 200, 50, 50),
    ("녹색(Avocado)", 100, 150, 50),
    ("파란색(Blueberry)", 50, 50, 200),
]

for desc, r, g, b in test_cases:
    rgb_input = torch.tensor([r/255.0, g/255.0, b/255.0], dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = loaded_model(rgb_input.unsqueeze(0))
        pred_idx = logits.argmax(1).item()
        pred_class = classes[pred_idx]
    print(f"{desc}: 예측={pred_class}")


=== 원본 이미지의 클래스별 RGB 통계 (증강 제외) ===

Apple Crimson Snow:
  평균 RGB: (147, 94, 92)
  범위: R[125~168], G[73~112], B[71~106]

Banana:
  평균 RGB: (218, 211, 190)
  범위: R[175~236], G[162~232], B[136~217]

Blueberry:
  평균 RGB: (113, 118, 124)
  범위: R[77~130], G[80~135], B[84~142]

Orange:
  평균 RGB: (174, 122, 73)
  범위: R[164~182], G[115~132], B[63~86]


=== 모델 예측 vs 학습 데이터 비교 ===
노란색(바나나 예상): 예측=Orange
빨간색(사과 예상): 예측=Orange
파란색(블루베리): 예측=Blueberry
주황색(오렌지): 예측=Orange
